### Look at vegetation carbon by biome

In [1]:
import numpy as np
import xarray as xr
import csv
import pandas as pd


import netCDF4 as nc4
import sys
import xesmf

import cartopy.crs as ccrs

import matplotlib.pyplot as plt
from matplotlib.pyplot import figure


import regrid_functions as rf

import xarray_funcs as xf

In [2]:
pftnames = ["broadleaf_evergreen_tropical_tree",
  "needleleaf_evergreen_extratrop_tree",
  "needleleaf_colddecid_extratrop_tree",
  "broadleaf_evergreen_extratrop_tree",
  "broadleaf_hydrodecid_tropical_tree",
  "broadleaf_colddecid_extratrop_tree",
  "broadleaf_evergreen_extratrop_shrub",
  "broadleaf_hydrodecid_extratrop_shrub",
  "broadleaf_colddecid_extratrop_shrub",
  " broadleaf_evergreen_arctic_shrub",
  " broadleaf_colddecid_arctic_shrub",
  "arctic_c3_grass",
  "cool_c3_grass",
  "c4_grass"]

In [3]:
biomes = ['Moist tropical forest (Americas)',
             'Moist tropical forest (Africa)',
             'Moist tropical forest (Asia)',
             'Tropical and subtropical dry forest, shrubland (Americas)',
             'Tropical and subtropical dry forest, shrubland (Africa)',
             'Tropical and subtropical dry forest, shrubland (Asia)',
             'Tropical and subtropical other vegetation (Americas)',
             'Tropical and subtropical other vegetation (Africa)',
             'Tropical and subtropical other vegetation (Asia)',
             'Temperate forest, shrubland (North America)',
             'Boreal forest, shrubland (North America)',
             'Tundra (North America)',
             'Temperate other vegetation (North America)',
             'Southern forest (South America)', 
             'Temperate forest, shrubland (Eurasia)',
             'Boreal forest, shrubland (Eurasia)',
             'Tundra (Eurasia)',
             'Temperate other vegetation (Eurasia)'
             ]

print(len(biomes))

18


In [4]:
fates_file = '/datalake/NS9188K/share/jessien/feb26/ne16_transient_1900.2026-02-06/run/ne16_transient_1900.2026-02-06.21stC.nc'
fates = xr.open_dataset(fates_file, decode_times=False)

FileNotFoundError: [Errno 2] No such file or directory: '/datalake/NS9188K/share/jessien/feb26/ne16_transient_1900.2026-02-06/run/ne16_transient_1900.2026-02-06.21stC.nc'

In [ ]:
weight_file = "/datalake/NS9560K/diagnostics/land_xesmf_diag_data/map_ne16pg3_to_1.9x2.5_nomask_scripgrids_c250425.nc"
regridder = rf.make_se_regridder(weight_file=weight_file)

### quick check of no comp PFT area 

In [ ]:
pfts = fates['FATES_NOCOMP_PATCHAREA_PF'].mean(dim='time')
pft_se = xr.DataArray(pfts)
pfts = rf.regrid_se_data(regridder, pft_se)
clevs=np.arange(0,0.3, 0.005)

extent = [65,180, -70, 0]

fig, axs = plt.subplots(nrows=4,ncols=4,
                        subplot_kw={'projection': ccrs.PlateCarree()},
                        figsize=(18,8))

axs = axs.flatten()


for i in range(14):
    cs= pfts.isel(fates_levpft=i).plot(levels=clevs,cmap='Greens', ax=axs[i], add_colorbar=True)
    axs[i].set_title(pftnames[i], fontsize=10)
    axs[i].coastlines()
    #axs[i].set_extent(extent)

In [ ]:
VEGC_PF = fates['FATES_VEGC_PF'].mean(dim='time')
vegc_PF_se = xr.DataArray(VEGC_PF)
vegc_pf = rf.regrid_se_data(regridder, vegc_PF_se)

In [ ]:
print(vegc_pf.lon.min().values, vegc_pf.lon.max().values)

In [ ]:
# convert 0 to 360 to -180 to 180
vegc_pf = vegc_pf.assign_coords(lon=(((vegc_pf.lon + 180) % 360) - 180)).sortby('lon')

In [ ]:
moist_trop_for=vegc_pf.sel(fates_levpft=1).sel(lat=slice(-70,70))

moist_trop_for_amer = moist_trop_for.sel(lon=slice(-175, -20))
moist_trop_for_asia = moist_trop_for.sel(lon=slice(62, 180))
moist_trop_for_afri= moist_trop_for.sel(lon=slice(-20,62)).sel(lat=slice(-70,38))

temp_for_shrub_euras = vegc_pf.sel(fates_levpft=2) + vegc_pf.sel(fates_levpft=3) + vegc_pf.sel(fates_levpft=4) + vegc_pf.sel(fates_levpft=5) + vegc_pf.sel(fates_levpft=6)  
temp_for_shrub_euras = temp_for_shrub_euras.sel(lat=slice(38,55)).sel(lon=slice(-20,180))

bor_for_shrub_euras = vegc_pf.sel(fates_levpft=2)+vegc_pf.sel(fates_levpft=3) + vegc_pf.sel(fates_levpft=6)
bor_for_shrub_euras = bor_for_shrub_euras.sel(lat=slice(55,90)).sel(lon=slice(-20,180))
#tmp1 = bor_for_shrub_euras
#tmp2 = bor_for_shrub_euras.sel(lon=slice(-180,-170))
#bor_for_shrub_euras = xr.concat([tmp1,tmp2],dim='lon').sortby('lon')

temp_other_euras = vegc_pf.sel(fates_levpft=7) + vegc_pf.sel(fates_levpft=8) + vegc_pf.sel(fates_levpft=9)+  vegc_pf.sel(fates_levpft=13) + vegc_pf.sel(fates_levpft=14)
temp_other_euras = temp_other_euras.sel(lat=slice(38,90)).sel(lon=slice(-20,180))

tundra_euras = vegc_pf.sel(fates_levpft=10) + vegc_pf.sel(fates_levpft=11) + vegc_pf.sel(fates_levpft=12)
tundra_euras = tundra_euras.sel(lat=slice(38,90)).sel(lon=slice(-20,180))

dry_for_shrub_afri = vegc_pf.sel(fates_levpft=5).sel(lat=slice(-70,38)).sel(lon=slice(-20,62))
other_afri = vegc_pf.sel(fates_levpft=4) + vegc_pf.sel(fates_levpft=8) + vegc_pf.sel(fates_levpft=9) + vegc_pf.sel(fates_levpft=13) + vegc_pf.sel(fates_levpft=14)
other_afri = other_afri.sel(lat=slice(-70,38)).sel(lon=slice(-20,62))

dry_for_shrub_asia = vegc_pf.sel(fates_levpft=5) + vegc_pf.sel(fates_levpft=7)+ vegc_pf.sel(fates_levpft=8)+ vegc_pf.sel(fates_levpft=9) 
dry_for_shrub_asia = dry_for_shrub_asia.sel(lat=slice(-70,38)).sel(lon=slice(62,180))
other_asia = vegc_pf.sel(fates_levpft=2) + vegc_pf.sel(fates_levpft=3) + vegc_pf.sel(fates_levpft=4) + vegc_pf.sel(fates_levpft=6) + vegc_pf.sel(fates_levpft=12) + vegc_pf.sel(fates_levpft=13) + vegc_pf.sel(fates_levpft=14)
other_asia = other_asia.sel(lat=slice(-70,38)).sel(lon=slice(62,180))

dry_for_shrub_amer = vegc_pf.sel(fates_levpft=5) 
dry_for_shrub_amer = dry_for_shrub_amer.sel(lat=slice(-70,70)).sel(lon=slice(-175, -20))

other_samer = vegc_pf.sel(fates_levpft=7) + vegc_pf.sel(fates_levpft=8) + vegc_pf.sel(fates_levpft=9) + vegc_pf.sel(fates_levpft=13) + vegc_pf.sel(fates_levpft=14)
other_samer = other_samer.sel(lat=slice(-70,20)).sel(lon=slice(-175, -20))

sfor_samer = vegc_pf.sel(fates_levpft=2) + vegc_pf.sel(fates_levpft=4) + vegc_pf.sel(fates_levpft=6) 
sfor_samer = sfor_samer.sel(lat=slice(-70,20)).sel(lon=slice(-175, -20))


other_namer = vegc_pf.sel(fates_levpft=7) + vegc_pf.sel(fates_levpft=8) + vegc_pf.sel(fates_levpft=9)+ vegc_pf.sel(fates_levpft=12) + vegc_pf.sel(fates_levpft=13)+ vegc_pf.sel(fates_levpft=14)
other_namer = other_namer.sel(lat=slice(20,90)).sel(lon=slice(-175, -20))

temp_amer = vegc_pf.sel(fates_levpft=2)+vegc_pf.sel(fates_levpft=3) +vegc_pf.sel(fates_levpft=4) + vegc_pf.sel(fates_levpft=6) 
temp_amer = temp_amer.sel(lat=slice(20,55)).sel(lon=slice(-175, -20))

bor_amer = vegc_pf.sel(fates_levpft=2) + vegc_pf.sel(fates_levpft=3) + vegc_pf.sel(fates_levpft=6)
bor_amer = bor_amer.sel(lat=slice(55,90)).sel(lon=slice(-175, -20))

tundra_amer = vegc_pf.sel(fates_levpft=10) + vegc_pf.sel(fates_levpft=11)
tundra_amer = tundra_amer.sel(lat=slice(0,90)).sel(lon=slice(-175, -20))


In [ ]:
# plot them to check they are all in the correct place
clevs=np.arange(0,10,0.25)
fig, axs = plt.subplots(nrows=6,ncols=3,
                        subplot_kw={'projection': ccrs.PlateCarree()},
                        figsize=(16,13))
axs=axs.flatten()

moist_trop_for_amer.plot(levels=clevs,cmap='Greens', ax=axs[0], add_colorbar=False)
moist_trop_for_afri.plot(levels=clevs,cmap='Greens', ax=axs[1], add_colorbar=False)
moist_trop_for_asia.plot(levels=clevs,cmap='Greens', ax=axs[2], add_colorbar=False)
dry_for_shrub_amer.plot(levels=clevs,cmap='Greens', ax=axs[3], add_colorbar=False)
dry_for_shrub_afri.plot(levels=clevs,cmap='Greens', ax=axs[4], add_colorbar=False)
dry_for_shrub_asia.plot(levels=clevs,cmap='Greens', ax=axs[5], add_colorbar=False)
other_samer.plot(levels=clevs,cmap='Greens', ax=axs[6], add_colorbar=False)
other_afri.plot(levels=clevs,cmap='Greens', ax=axs[7], add_colorbar=False)
other_asia.plot(levels=clevs,cmap='Greens', ax=axs[8], add_colorbar=False)
temp_amer.plot(levels=clevs,cmap='Greens', ax=axs[9], add_colorbar=False)
bor_amer.plot(levels=clevs,cmap='Greens', ax=axs[10], add_colorbar=False)
tundra_amer.plot(levels=clevs,cmap='Greens', ax=axs[11], add_colorbar=False)
other_namer.plot(levels=clevs,cmap='Greens', ax=axs[12], add_colorbar=False)
sfor_samer.plot(levels=clevs,cmap='Greens', ax=axs[13], add_colorbar=False)
temp_for_shrub_euras.plot(levels=clevs,cmap='Greens', ax=axs[14], add_colorbar=False)
bor_for_shrub_euras.plot(levels=clevs,cmap='Greens', ax=axs[15], add_colorbar=False)
tundra_euras.plot(levels=clevs,cmap='Greens', ax=axs[16], add_colorbar=False)
temp_other_euras.plot(levels=clevs,cmap='Greens', ax=axs[17], add_colorbar=False)

for i in range(len(biomes)): 
    axs[i].set_title(biomes[i], fontsize=9)
    axs[i].coastlines() 
    
fig.subplots_adjust(right=0.8)
cbar_ax = fig.add_axes([0.85, 0.2, 0.025, 0.65])
fig.colorbar(axs[0].collections[0], cax=cbar_ax, label='Vegetation carbon (kgC/m2)')
plt.show()

In [ ]:
# for each region get total carbon in Pg C
km2_to_m2 = 1e6
kg_to_Gt = 1e-12
area = fates['area']
area = xr.DataArray(area)
area = rf.regrid_se_data(regridder, area)
area = area.assign_coords(lon=(((area.lon + 180) % 360) - 180)).sortby('lon')
landfrac = fates['landfrac']
landfrac = xr.DataArray(landfrac)
landfrac = rf.regrid_se_data(regridder, landfrac)
landfrac = landfrac.assign_coords(lon=(((landfrac.lon + 180) % 360) - 180)).sortby('lon')
fatesfrac = fates['FATES_FRACTION'].mean(dim='time')
fatesfrac = xr.DataArray(fatesfrac)
fatesfrac = rf.regrid_se_data(regridder, fatesfrac)
fatesfrac = fatesfrac.assign_coords(lon=(((fatesfrac.lon + 180) % 360) - 180)).sortby('lon')
area_land = area * landfrac * km2_to_m2 




In [ ]:
total_moist_trop_for_amer = (moist_trop_for_amer * area_land * kg_to_Gt * fatesfrac).sum(dim=['lat','lon']).values
total_moist_trop_for_asia = (moist_trop_for_asia * area_land * kg_to_Gt * fatesfrac).sum(dim=['lat','lon']).values
total_moist_trop_for_afri = (moist_trop_for_afri * area_land * kg_to_Gt * fatesfrac).sum(dim=['lat','lon']).values
total_temp_for_shrub_euras = (temp_for_shrub_euras * area_land * kg_to_Gt * fatesfrac).sum(dim=['lat','lon']).values
total_bor_for_shrub_euras = (bor_for_shrub_euras * area_land * kg_to_Gt * fatesfrac).sum(dim=['lat','lon']).values
total_dry_for_shrub_afri = (dry_for_shrub_afri * area_land * kg_to_Gt * fatesfrac).sum(dim=['lat','lon']).values
total_temp_other_euras = (temp_other_euras * area_land * kg_to_Gt * fatesfrac).sum(dim=['lat','lon']).values
total_dry_for_shrub_asia = (dry_for_shrub_asia * area_land * kg_to_Gt * fatesfrac).sum(dim=['lat','lon']).values
total_dry_for_shrub_amer = (dry_for_shrub_amer * area_land * kg_to_Gt * fatesfrac).sum(dim=['lat','lon']).values
total_other_afri = (other_afri * area_land * kg_to_Gt * fatesfrac).sum(dim=['lat','lon']).values
total_bor_amer = (bor_amer * area_land * kg_to_Gt * fatesfrac).sum(dim=['lat','lon']).values
total_tundra_euras = (tundra_euras * area_land * kg_to_Gt * fatesfrac).sum(dim=['lat','lon']).values
total_temp_amer = (temp_amer * area_land * kg_to_Gt * fatesfrac).sum(dim=['lat','lon']).values
total_other_asia = (other_asia * area_land * kg_to_Gt * fatesfrac).sum(dim=['lat','lon']).values
total_other_namer = (other_namer * area_land * kg_to_Gt * fatesfrac).sum(dim=['lat','lon']).values
total_other_samer = (other_samer * area_land * kg_to_Gt * fatesfrac).sum(dim=['lat','lon']).values
total_tundra_amer = (tundra_amer * area_land * kg_to_Gt * fatesfrac).sum(dim=['lat','lon']).values
total_sfor_samer = (sfor_samer * area_land * kg_to_Gt * fatesfrac).sum(dim=['lat','lon']).values

In [ ]:
values = [total_moist_trop_for_amer,
          total_moist_trop_for_afri, 
total_moist_trop_for_asia ,
total_dry_for_shrub_amer ,
total_dry_for_shrub_afri,
total_dry_for_shrub_asia, 
total_other_samer ,
total_other_afri ,
total_other_asia ,
total_temp_amer ,
total_bor_amer,
total_tundra_amer,
total_other_namer ,
total_sfor_samer,
total_temp_for_shrub_euras, 
total_bor_for_shrub_euras,
total_tundra_euras, 
total_temp_other_euras 
]

print(values)
print(len(values))

In [ ]:
xu = [81.37, 36.1, 36.78, 15.16, 21.41, 17.97, 5.77, 12.35, 7.36, 30.94, 11.93, 3.57, 7.66, 2.47, 33.12, 27.19, 8.0, 19.28]
uncert = [0.36, 0.26, 0.8, 0.2, 0.22, 0.54, 0.1, 0.13, 0.13, 0.51, 0.12, 0.06, 0.12, 0.08, 0.45, 0.20, 0.09, 0.29]

print(len(biomes))
print(len(xu))
print(len(uncert))

data = {
    'Biome': biomes,
    'Xu': xu, 
    'Uncertainty': uncert,
    'FATES': values
}

xu_data = pd.DataFrame(data)

xu_data.head(3)

In [ ]:
# make a bar plot
plt.figure(figsize=(10, 10))
                        


categories = biomes

plt.bar(categories, xu)
plt.xlabel('Biome', fontsize=12)
plt.ylabel('Veg C (Pg C)', fontsize=12)
plt.title('Global Veg C by Biome', fontsize=14)
plt.xticks(rotation=45, ha='right') # 'ha' is shorthand for horizontalalignment

# Adjust layout to prevent labels from being cut off
plt.tight_layout() 

plt.show()

In [ ]:
# total by biome
np.sum(values)

In [ ]:
# total by globe
total_globe = (vegc_pf.sum(dim='fates_levpft') * area_land * kg_to_Gt * fatesfrac).sum(dim=['lat','lon']).values
print(total_globe)


In [ ]:
 # Set up the bar positions
x = np.arange(len(biomes))
width = 0.35  # width of each bar

fig, ax = plt.subplots(figsize=(14, 9))

# Create bars for Xu and FATES with offset positions
bars1 = ax.bar(x - width/2, xu, width, label='Xu et al (2021)', alpha=0.8)
bars2 = ax.bar(x + width/2, values, width, label='FATES', alpha=0.8)

# Customize the plot
ax.set_xlabel('Biome', fontsize=12)
ax.set_ylabel('Veg C (Pg C)', fontsize=12)
ax.set_title('Global Veg C by Biome: Xu vs FATES', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(biomes, rotation=45, ha='right')
ax.legend()

plt.tight_layout()
plt.show()